# Lab 1 · Exercise B — LHCb mass peak and the central limit theorem
**B1.** Simplified dataset modelled on an LHCb analysis: histogram of a B-meson invariant mass (`gallery/data/lhcb-b0_invariant_mass.root`, histogram `massaB0`).
**B2.** Sum of N uniform variables: where does the Gaussian approximation stop working?

In [ ]:
import ROOT
# interactive drawing in the notebook (needed on Binder)
%jsroot on
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## B1 · Breit–Wigner fit of a binned mass spectrum
1. Import the histogram as a `RooDataHist`. 2. Fit a Breit–Wigner. 3. Add a Gaussian resolution and refit.
4. Compare the mass with the PDG value (≈ 5.367 GeV for the $B^0_s$).

In [ ]:
f = ROOT.TFile.Open("../../gallery/data/lhcb-b0_invariant_mass.root")
h = f.Get("massaB0")
h.SetDirectory(0)
f.Close()

mass = ROOT.RooRealVar("mass", "mass [GeV]", 5.0, 6.0)
dh   = None   # TODO: RooDataHist from h, with observable mass:  ROOT.RooDataHist("dh","dh", ROOT.RooArgList(mass), h)

mean_bw  = ROOT.RooRealVar("mean_bw", "mass", 5.36, 5.2, 5.5)
width_bw = ROOT.RooRealVar("width_bw", "width", 0.1, 0.001, 0.5)
bw       = None   # TODO: RooBreitWigner(mass, mean_bw, width_bw)
res_bw   = None   # TODO: binned ML fit (Save=True, PrintLevel=-1)

frame_b = mass.frame(Title="Breit-Wigner fit")
# TODO: plot dh, then bw
c_b = ROOT.TCanvas("c_b", "c_b", 800, 500)
frame_b.Draw()
c_b.Draw()
mean_bw.Print(); width_bw.Print()

In [ ]:
check("BW fit converged", res_bw is not None and res_bw.status() == 0)
check("mass within 30 MeV of PDG (5.367 GeV)", abs(mean_bw.getVal() - 5.367) < 0.03)

### Add a Gaussian resolution
`RooVoigtian` is the analytic convolution Breit–Wigner ⊗ Gaussian.
Fix the resolution at σ_res = 30 MeV (it cannot be floated together with the width: why not?) and refit.

In [ ]:
mean_v  = ROOT.RooRealVar("mean_v", "mass", 5.36, 5.2, 5.5)
width_v = ROOT.RooRealVar("width_v", "width", 0.1, 0.001, 0.5)
sig_res = ROOT.RooRealVar("sig_res", "resolution", 0.03)   # fixed value, no range
voigt   = None   # TODO: RooVoigtian(mass, mean_v, width_v, sig_res)
res_v   = None   # TODO: fit it to dh

frame_v = mass.frame(Title="Voigtian = BW (x) Gaussian")
# TODO: plot dh and voigt
c_v = ROOT.TCanvas("c_v", "c_v", 800, 500)
frame_v.Draw()
c_v.Draw()
print("BW width:", width_bw.getVal(), " Voigtian BW width:", width_v.getVal())

In [ ]:
check("Voigtian fit converged", res_v is not None and res_v.status() == 0)

**Questions B1.** (a) How far is your mass from the PDG value, in units of the fit error?
(b) The natural width of the $B^0_s$ is of order 10⁻¹² GeV. Is the fitted width physical? What does it measure?
(c) Why do width and resolution cannot be fitted together here?

## B2 · The central limit theorem
The sum of many independent random variables approaches a **Gaussian**, whatever the distribution of each one. We check it with RooFit, following the macro `central_limit_10.cpp`: **10 independent variables**, each **uniform between 0 and 1**, and the sums of the first 2, 3 and 5 of them.
1. Create the variables and the model. 2. Generate 10 000 events. 3. Add the sums as new **columns** of the dataset. 4. Fit a Gaussian to the sum of 5 variables. 5. Plot.

In [ ]:
variables = []                  # the 10 variables, each between 0 and 1
pdfs = []                       # the 10 uniform pdfs
var_set = ROOT.RooArgSet()      # all the variables together
pdf_list = ROOT.RooArgList()    # all the pdfs together

for i in range(10):
    v = ROOT.RooRealVar("var%d" % i, "var%d" % i, 0, 1)
    u = ROOT.RooUniform("model%d" % i, "model%d" % i, v)
    variables.append(v)
    pdfs.append(u)
    var_set.add(v)
    pdf_list.add(u)

# The 10 variables are independent: the model is the product of the 10 pdfs
model = None    # TODO: ROOT.RooProdPdf("model", "model", pdf_list)

Generate the events: `generate` draws values of **all** the variables of `var_set`, from the model.

In [ ]:
data = None     # TODO: generate 10000 events of all the variables: model.generate(var_set, 10000)
data.Print()

Now the **sums**. A `RooFormulaVar` is a formula of other variables (`@0`, `@1`, ... are the variables of the list). `addColumn` calculates it for every event and stores it in the dataset.

In [ ]:
# A formula with the sum of 2 variables. The "-1" only centres the distribution at zero.
f2 = ROOT.RooFormulaVar("f2", "@0+@1-1.", ROOT.RooArgList(variables[0], variables[1]))
sum2 = data.addColumn(f2)       # a new column of the dataset

# The same for 3 variables
f3 = ROOT.RooFormulaVar("f3", "@0+@1+@2-1.5", ROOT.RooArgList(variables[0], variables[1], variables[2]))
sum3 = data.addColumn(f3)

# And for 5 variables: here there is no shift
f5 = None       # TODO: ROOT.RooFormulaVar("f5", "@0+@1+@2+@3+@4", ROOT.RooArgList(variables[0], variables[1], variables[2], variables[3], variables[4]))
sum5 = None     # TODO: data.addColumn(f5)

data.Print()

Fit a Gaussian to the sum of 5 variables. Only the column `sum5` is in common between the Gaussian and the dataset: it is the **observable**. The other columns are not used.

In [ ]:
# A Gaussian for the sum of 5 variables. The observable is the column sum5: m and s are the parameters.
m = ROOT.RooRealVar("m", "m", 3, 0, 6)
s = ROOT.RooRealVar("s", "s", 2, 0.5, 5)
gaus = ROOT.RooGaussian("gaus", "gaus", sum5, m, s)

res = None      # TODO: fit the Gaussian to the data: gaus.fitTo(data, Save=True, PrintLevel=-1)

# Each uniform variable has mean 1/2 and variance 1/12: the sum of 5 has mean 5/2 and variance 5/12
check("the fit converged", res is not None and res.status() == 0)
check("the mean is close to 5/2", abs(m.getVal() - 2.5) < 0.05)
check("sigma is close to sqrt(5/12)", abs(s.getVal() - (5.0/12.0)**0.5) < 0.05)
print("m = %.3f +- %.3f   s = %.3f +- %.3f" % (m.getVal(), m.getError(), s.getVal(), s.getError()))

Plot the four distributions.

In [ ]:
plot  = variables[0].frame(ROOT.RooFit.Bins(100), ROOT.RooFit.Title("Random variable"), ROOT.RooFit.Range(-5., 5.))
data.plotOn(plot)

plot2 = sum2.frame(ROOT.RooFit.Bins(100), ROOT.RooFit.Title("Sum of 2 random variables"), ROOT.RooFit.Range(-5., 5.))
data.plotOn(plot2)

plot3 = sum3.frame(ROOT.RooFit.Bins(100), ROOT.RooFit.Title("Sum of 3 random variables"), ROOT.RooFit.Range(-5., 5.))
data.plotOn(plot3)

plot5 = sum5.frame(ROOT.RooFit.Bins(100), ROOT.RooFit.Title("Sum of 5 random variables"), ROOT.RooFit.Range(-5., 5.))
data.plotOn(plot5)
gaus.plotOn(plot5)             # the fitted Gaussian

c = ROOT.TCanvas("c", "central limit theorem", 900, 700)
c.Divide(2, 2)
c.cd(1); plot.Draw()
c.cd(2); plot2.Draw()
c.cd(3); plot3.Draw()
c.cd(4); plot5.Draw()
c.Draw()

**Questions B2.** (a) How does the shape change from one variable to the sum of 2, of 3 and of 5? Which one is already close to a Gaussian?
(b) Where do the fitted mean 2.5 and the width $\sqrt{5/12}\approx0.645$ come from?
(c) The sum of 2 variables is a triangle with hard edges: a Gaussian never ends. What does this tell you about the **tails** of a Gaussian approximation, and about quoting a "5σ" result from it?